# ThermoNet — all-cohort training

Train a **fresh model** on the single-, two-, and 3–10-cube datasets. The old checkpoint provides split metadata only; its weights are not loaded. Select **Runtime → Change runtime type → GPU**.

Prepare an archive from the ThermoFlow project folder, where the NPZ contents are available locally:

```bash
cp ~/Downloads/best.pt best_single.pt
zip -r -0 thermoflow_all.zip thermonet.py train_thermonet.py train_thermonet_all.py best_single.pt datasets/single_cube datasets/two_cubes datasets/mixed_cubes
```

Upload `thermoflow_all.zip` to `My Drive/ThermoFlow/`. The archive needs the three Python files, the original single-cube checkpoint, and the three dataset folders. If iCloud files are not downloaded, create the archive where the data is local (for example, on Moana after copying the scripts/checkpoint there).


In [ ]:
import torch
import scipy
print("PyTorch:", torch.__version__, "SciPy:", scipy.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime, reconnect, and rerun this cell.")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import os
import shutil
import zipfile

archive = Path("/content/drive/MyDrive/ThermoFlow/thermoflow_all.zip")
project = Path("/content/ThermoFlowAll")
run = Path("/content/drive/MyDrive/ThermoFlow/runs/all_cohorts")
if not archive.is_file():
    raise FileNotFoundError(f"Upload the archive to {archive}")
local_archive = Path("/content/thermoflow_all.zip")
shutil.copyfile(archive, local_archive)
project.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(local_archive) as bundle:
    for name in bundle.namelist():
        target = (project / name).resolve()
        if project.resolve() not in target.parents:
            raise ValueError(f"Unexpected archive path: {name}")
    bundle.extractall(project)
os.chdir(project)
for cohort in ("single_cube", "two_cubes", "mixed_cubes"):
    print(cohort, len(list((project / "datasets" / cohort).glob("sample_*.npz"))))
if not (project / "best_single.pt").is_file():
    raise FileNotFoundError("Include best_single.pt to preserve the original single-cube split")


## Train or resume

The original single-cube partition is preserved exactly. Two- and multi-cube specimens are split by **unique geometry**, stratified by true cube count, with approximately 70/15/15 proportions. Duplicate interiors cannot cross partitions. Relative paths include the cohort name to prevent filename collisions.

Normalization and class weights are fitted on the combined training partition. Training uses the same weighted BCE + soft Dice objective and architecture as the single-cube baseline. Checkpoints and metrics go to a **new Drive folder**, preserving the earlier run. Caching all 7,500 default specimens uses about 1.5 GiB for arrays, plus Python and loader overhead; remove `--cache` if RAM is constrained.


In [ ]:
import subprocess
import sys

command = [sys.executable, "-u", "train_thermonet_all.py",
           "--data", "datasets", "--single-checkpoint", "best_single.pt",
           "--output", str(run), "--epochs", "100", "--batch-size", "16",
           "--num-workers", "2", "--device", "cuda", "--cache"]
if (run / "last.pt").is_file():
    command += ["--resume", str(run / "last.pt")]
subprocess.run(command, check=True)


In [ ]:
import json
import matplotlib.pyplot as plt

history = json.loads((run / "history.json").read_text())
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
epochs = [r["epoch"] for r in history]
for split in ("train", "validation"):
    axes[0].plot(epochs, [r[split]["loss"] for r in history], label=split)
    axes[1].plot(epochs, [r[split]["dice"] for r in history], label=split)
axes[0].set(xlabel="Epoch", ylabel="Weighted BCE + soft Dice loss")
axes[1].set(xlabel="Epoch", ylabel="Thresholded 3D Dice", ylim=(0, 1))
for ax in axes:
    ax.legend()
    ax.grid(alpha=.2)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
results = json.loads((run / "test_metrics.json").read_text())
print("Best validation epoch:", results["best_epoch"])
print("Original single split preserved:", results["original_single_split_preserved"])
display(pd.DataFrame(results["by_cohort"]).T)
display(pd.DataFrame(results["by_true_cube_count"]).T)


## Interpretation

Final metrics use the best validation checkpoint and are broken down by cohort and true cube count. Voxel Dice/IoU are averaged per specimen. Object recall and precision pool one-to-one component matches at IoU ≥ 0.5, using a fixed probability threshold of 0.5.

The earlier six visualization examples may now belong to training or validation. They can illustrate reconstructions, but use the new **held-out test set** for performance claims. The single-cube holdout stays unchanged for comparison with the earlier baseline.

Artifacts: `manifest.json`, split lists, `normalization.json`, `best.pt`, `last.pt`, `history.json`, `test_metrics.json`, and `test_per_sample.csv`. The manifest records cohort-relative filenames, content/geometry hashes, and the per-partition cube-count distribution.
